# 不平衡处理

**负责人：** 6 号　**阶段：** 4 分析

比较几种策略，把最好的方案分享给全组。

## 任务

标"（选做）"的任务在必做部分完成后再做。

- [x] 必做：不处理、class_weight、随机过采样三者对比（重采样只能在训练折内做）
- [x] （选做）SMOTE
- [ ] （选做）Focal loss。它改的是神经网络的损失，这一版没有放进对比
- [x] 看 macro-F1 和各类别召回率
- [x] 每次实验先用 `new_run_id()` 生成 run_id，评分记录和预测文件都带上同一个 run_id

## 产出

- `results/imbalance.csv`
- `figures/analysis_imbalance.png`

In [ ]:
import sys
from pathlib import Path

# 向上查找仓库根目录（含 src/config.py 的文件夹），在任何子文件夹里打开都能导入 src
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录：请在 clone 下来的仓库里启动 Jupyter（Colab 先 %cd 进仓库）")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED                 # 路径和常量
from src.data import load_clean, load_raw, load_splits                      # 读取数据和划分
from src.evaluate import evaluate, new_run_id, plot_confusion, save_result  # 统一评分
from src.cache import load_embedding, save_embedding, save_preds            # 嵌入和预测结果的读写

print("仓库根目录:", ROOT)
print("DATA_ROOT:", DATA_ROOT)

## Approach

The four strategies are compared on one fixed classifier, so the gap is the imbalance treatment and not a different model. The classifier is logistic regression (C = 1, no tuning) on fold-wise FastText mean vectors. TF-IDF stays reserved for the baseline.

Random oversampling and SMOTE run only on the training portion of the current fold. The validation fold and the test split keep their original rows. The strategy is chosen by out-of-fold macro-F1. If another strategy is within 0.005, the simpler one wins, in this order: no treatment, `class_weight`, random oversampling, SMOTE. The test split is scored once, for the chosen strategy only.

Recorded CV macro-F1 on `folds_v1`: no treatment 0.509, `class_weight` 0.527, random oversampling 0.525, SMOTE 0.526. `class_weight` also lifts Negative recall from 0.35 to 0.46. Recommendation for the group: train with `class_weight="balanced"` (for a neural net, weight the loss by inverse class frequency). Oversampling does not add enough to be the default. These absolute scores sit below the TF-IDF baseline (0.582) because FastText is a weak embedding on 1,500 tweets; the comparison is about the treatment, not about beating the baseline.

In [ ]:
from Member6_file.m6_imbalance import run_imbalance

results = run_imbalance()
results.loc[:, ["strategy", "split", "macro_f1", "recall_neutral", "recall_positive", "recall_negative", "run_id"]]